In [ ]:
from openai import OpenAI

#Connecct to LM Studio Local server
client = OpenAI(
    base_url="http://127.0.0.1:1234",
    api_key="lm-studio"
)

In [ ]:
import keyboard
from langchain_community.chat_message_histories import ChatMessageHistory

# Initialize the history object to keep track of the discussion
history = ChatMessageHistory()

In [ ]:
while True:
    query = input("\nUser (Enter to Continue || Type 'exit' or press ESC to exit): ")

    # Check for the ESC character (\x1b) OR a simple 'exit' command
    if keyboard.is_pressed('Esc') or query.lower() == 'exit':
        print("\nExiting...")
        print("Program Finished.")
        break
    print("User Input: " + query)
    print('- ' * 50)
    if not query.strip():
        continue
    # Add user message to LangChain history
    history.add_user_message(query)

    # Convert LangChain 'human/ai' roles to OpenAI 'user/assistant' roles
    messages_with_history = []
    for m in history.messages:
        # LangChain uses 'human' for user and 'ai' for assistant
        role = "user" if m.type == "human" else "assistant"
        messages_with_history.append({"role": role, "content": m.content})

    # Ensured stream is called only once with the correctly mapped history
    stream = client.chat.completions.create(
        model="google/gemma-4-e4b",
        messages=messages_with_history,
        stream=True
    )
    full_ai_response = ""       # Temporary variable to capture the full AI response for memory

    for chunk in stream:
        if chunk.choices[0].delta.content:
            content = chunk.choices[0].delta.content
            print(content, end='', flush=True)  # Print the content as it streams
            full_ai_response += content  # Append to the full response

 # Save the AI response back to LangChain history
    history.add_ai_message(full_ai_response)
    print('\n ')     # Print a newline for readability
    print('=' * 100)
    print(' ')